# Cálculo de Volumen y Selección de Depósito GLP



In [1]:
import pandas as pd
import numpy as np

# 1. Parámetros y Constantes (Fuente: Proyecto/Anotaciones/calculo_volumen_deposito.md)
PCS_PROPANO = 13.95  # kWh/kg
DENSIDAD_LIQ = 506.0  # kg/m3 a 20°C
LLENADO_MAX = 0.85
RESERVA_MIN = 0.20
FRACCION_UTIL = LLENADO_MAX - RESERVA_MIN
AUTONOMIA_DIAS = 30  # Requerido por TAREA 3
TEMP_DISENO = -5.0   # IDAE Percentil 99,6% para León
PRESION_SERVICIO = 2.0 # bar (Condición de comprobación solicitada)

print(f"Fracción útil adoptada: {FRACCION_UTIL:.2f}")
print(f"Autonomía de diseño: {AUTONOMIA_DIAS} días")
print(f"Temperatura exterior de diseño: {TEMP_DISENO} ºC")
print(f"Presión de servicio para comprobación: {PRESION_SERVICIO} bar")

Fracción útil adoptada: 0.65
Autonomía de diseño: 30 días
Temperatura exterior de diseño: -5.0 ºC
Presión de servicio para comprobación: 2.0 bar


In [2]:
# 2. Datos de Consumidores (Fuente: Proyecto/Datos.md)
consumidores = [
    {"nombre": "Horno secado 1", "potencia_kw": 60, "horas_dia": 12},
    {"nombre": "Horno secado 2", "potencia_kw": 60, "horas_dia": 12},
    {"nombre": "Caldera vapor", "potencia_kw": 500, "horas_dia": 10},
    {"nombre": "Caldera agua caliente", "potencia_kw": 300, "horas_dia": 8},
    {"nombre": "Horno fusion", "potencia_kw": 700, "horas_dia": 4},
    {"nombre": "Horno decapado", "potencia_kw": 1000, "horas_dia": 6}
]

df_cons = pd.DataFrame(consumidores)
df_cons['energia_diaria_kwh'] = df_cons['potencia_kw'] * df_cons['horas_dia']

energia_total_dia = df_cons['energia_diaria_kwh'].sum()
potencia_max_simultanea = df_cons['potencia_kw'].sum()

print(f"Energía total diaria: {energia_total_dia} kWh/d")
print(f"Potencia máxima simultánea (S=1): {potencia_max_simultanea} kW")

Energía total diaria: 17640 kWh/d
Potencia máxima simultánea (S=1): 2620 kW


In [3]:
# 3. Cálculo de Consumo Másico y Volumétrico
m_dia = energia_total_dia / PCS_PROPANO
v_liq_dia = m_dia / DENSIDAD_LIQ  # m3/día

v_geom_min = (v_liq_dia * AUTONOMIA_DIAS) / FRACCION_UTIL

print(f"Consumo diario: {m_dia:.2f} kg/día")
print(f"Volumen líquido diario: {v_liq_dia:.3f} m3/día")
print(f"--- Volumen geométrico mínimo requerido: {v_geom_min:.2f} m3 ---")

Consumo diario: 1264.52 kg/día
Volumen líquido diario: 2.499 m3/día
--- Volumen geométrico mínimo requerido: 115.34 m3 ---


In [4]:
# 4. Selección del Depósito Comercial (Nueva Configuración por Distancias)
df_dep = pd.read_csv('datos/tabla_caracteristicas_secadores.csv')
df_vap_table = pd.read_csv('datos/caudal_vaporizacion.csv')
df_vapi_table = pd.read_csv('datos/deposito_vaporizador_interno.csv')

# Datos de la nueva selección
dep_grande_ref = 'LP46A-22'
dep_peque_ref = 'LP26A-22'
n_grande = 1
n_peque = 3

v_grande = df_dep[df_dep['Modelo Ref.'] == dep_grande_ref]['Capacidad nominal (litros)'].values[0]
v_peque = df_dep[df_dep['Modelo Ref.'] == dep_peque_ref]['Capacidad nominal (litros)'].values[0]
v_total = (v_grande * n_grande) + (v_peque * n_peque)

print(f"Configuración de Almacenamiento: {n_grande} x {dep_grande_ref} + {n_peque} x {dep_peque_ref}")
print(f"Capacidad total: {v_total} litros")

Configuración de Almacenamiento: 1 x LP46A-22 + 3 x LP26A-22
Capacidad total: 125100 litros


In [5]:
# 5. Verificación de Vaporización y Selección de Vaporizador Forzado
caudal_nec_kgh = potencia_max_simultanea / PCS_PROPANO
print(f"Demanda punta necesaria: {caudal_nec_kgh:.2f} kg/h")

# 5.1. Comprobación de Vaporización Natural (al 20% de llenado)
vol_grande_m3 = 46.2
vol_peque_m3 = 26.3

vap_grande = df_vap_table[(abs(df_vap_table['Volum. m3'] - vol_grande_m3) < 0.01) & (df_vap_table['Pres. bar'] == PRESION_SERVICIO)]
vap_peque = df_vap_table[(abs(df_vap_table['Volum. m3'] - vol_peque_m3) < 0.01) & (df_vap_table['Pres. bar'] == PRESION_SERVICIO)]

q_nat_grande = vap_grande['Caudal Aéreo -5°C'].values[0]
q_nat_peque_total = vap_peque['Caudal Aéreo -5°C'].values[0] * n_peque
q_nat_total = q_nat_grande + q_nat_peque_total

print(f"Vaporización natural {dep_grande_ref}: {q_nat_grande} kg/h")
print(f"Vaporización natural {dep_peque_ref} ({n_peque} uds): {q_nat_peque_total:.1f} kg/h")
print(f"Vaporización natural total (-5ºC, 2 bar, 20% llenado): {q_nat_total:.1f} kg/h")

if q_nat_total < caudal_nec_kgh:
    deficit = caudal_nec_kgh - q_nat_total
    print(f"Déficit de vaporización natural: {deficit:.2f} kg/h")
    
    # 5.2. Selección de Vaporizador Interno
    # Solo se requiere en el depósito más grande para cubrir el déficit total.
    capacidades_vapi = {"VIA 150": 150, "VIA 300": 300, "VIB 500": 500}
    potencias_caldera = {"VIA 150": 17.5, "VIA 300": 35, "VIB 500": 58}
    
    vapi_seleccionado = None
    for mod, cap in capacidades_vapi.items():
        if cap >= deficit:
            vapi_seleccionado = mod
            break
    
    if vapi_seleccionado:
        print(f"\n--- Selección de Vaporización Forzada ---")
        print(f"Vaporizador Interno Seleccionado (en {dep_grande_ref.replace('LP', 'LPVI')}): {vapi_seleccionado}")
        print(f"Capacidad forzada: {capacidades_vapi[vapi_seleccionado]} kg/h")
        print(f"Potencia de caldera requerida: {potencias_caldera[vapi_seleccionado]} kW")
        print(f"Vaporización Total (Nat + Forz): {q_nat_total + capacidades_vapi[vapi_seleccionado]:.2f} kg/h")
        print(f"RESULTADO: Suministro garantizado mediante sistema mixto (Vaporizador en el depósito de {vol_grande_m3}m3).")
    else:
        print("No se encontró un vaporizador interno suficiente.")
else:
    print("Vaporización natural suficiente.")

Demanda punta necesaria: 187.81 kg/h
Vaporización natural LP46A-22: 47.0 kg/h
Vaporización natural LP26A-22 (3 uds): 84.3 kg/h
Vaporización natural total (-5ºC, 2 bar, 20% llenado): 131.3 kg/h
Déficit de vaporización natural: 56.51 kg/h

--- Selección de Vaporización Forzada ---
Vaporizador Interno Seleccionado (en LPVI46A-22): VIA 150
Capacidad forzada: 150 kg/h
Potencia de caldera requerida: 17.5 kW
Vaporización Total (Nat + Forz): 281.30 kg/h
RESULTADO: Suministro garantizado mediante sistema mixto (Vaporizador en el depósito de 46.2m3).


## 6. Potencia termica del armario de calefaccion

El armario de calefaccion aporta el calor al circuito cerrado de agua que alimenta el serpentín interno del vaporizador forzado. Por tanto, su potencia minima se fija por la potencia de caldera requerida por el modelo de vaporizador seleccionado.

In [6]:
# 6. Calculo de potencia termica del armario de calefaccion
armario_calefaccion = 'VPC30C'
potencia_nominal_armario_kw = 45.0  # kW, caldera integrada en VPC30C según ficha tecnica consultada en GLP

if 'vapi_seleccionado' in globals() and vapi_seleccionado:
    capacidad_forzada_kgh = capacidades_vapi[vapi_seleccionado]
    potencia_minima_armario_kw = potencias_caldera[vapi_seleccionado]
    margen_vaporizacion_kgh = (q_nat_total + capacidad_forzada_kgh) - caudal_nec_kgh
    margen_potencia_armario_kw = potencia_nominal_armario_kw - potencia_minima_armario_kw

    print('--- Potencia termica del armario de calefaccion ---')
    print(f'Demanda punta de GLP: {caudal_nec_kgh:.2f} kg/h')
    print(f'Vaporizacion natural disponible: {q_nat_total:.1f} kg/h')
    print(f'Deficit que obliga a vaporizacion forzada: {deficit:.2f} kg/h')
    print(f'Vaporizador interno seleccionado: {vapi_seleccionado}')
    print(f'Capacidad forzada del vaporizador: {capacidad_forzada_kgh:.0f} kg/h')
    print(f'Armario de calefaccion asociado: {armario_calefaccion}')
    print(f'Potencia nominal del armario: {potencia_nominal_armario_kw:.1f} kW')
    print(f'Potencia termica minima requerida: {potencia_minima_armario_kw:.1f} kW')
    print(f'Margen de potencia del armario: {margen_potencia_armario_kw:.1f} kW')
    print(f'Margen total de vaporizacion: {margen_vaporizacion_kgh:.2f} kg/h')
    print(f'Comprobacion: {potencia_nominal_armario_kw:.1f} kW >= {potencia_minima_armario_kw:.1f} kW. Armario suficiente.')
else:
    potencia_minima_armario_kw = 0.0
    print('No se requiere potencia de armario: la vaporizacion natural resulta suficiente.')

--- Potencia termica del armario de calefaccion ---
Demanda punta de GLP: 187.81 kg/h
Vaporizacion natural disponible: 131.3 kg/h
Deficit que obliga a vaporizacion forzada: 56.51 kg/h
Vaporizador interno seleccionado: VIA 150
Capacidad forzada del vaporizador: 150 kg/h
Armario de calefaccion asociado: VPC30C
Potencia nominal del armario: 45.0 kW
Potencia termica minima requerida: 17.5 kW
Margen de potencia del armario: 27.5 kW
Margen total de vaporizacion: 93.49 kg/h
Comprobacion: 45.0 kW >= 17.5 kW. Armario suficiente.


## 7. Dimensionado de tuberías GLP

Se ejecuta el dimensionado por tramos con Renouard en media presión, longitud de cálculo iterativa por accesorios y verificación de velocidad. La salida documental técnica se consolida únicamente en `Proyecto/Anotaciones/dimensionado_tuberias_glp.md`.

In [7]:
# 7.1. Ejecución del dimensionado por tramos en la propia libreta
from pathlib import Path
from math import sqrt
from functools import lru_cache

P_ATM_BAR = 1.01325
P_INICIAL_REL_BAR = 1.70
P_INICIAL_ABS_BAR = P_INICIAL_REL_BAR + P_ATM_BAR
CAIDA_MAXIMA_FRACCION = 0.05
P_MIN_REL_BAR = P_INICIAL_REL_BAR * (1.0 - CAIDA_MAXIMA_FRACCION)
P_MIN_ABS_BAR = P_MIN_REL_BAR + P_ATM_BAR
DC_PROPANO = 1.16
DENSIDAD_PROPANO_GAS_KG_M3 = 1.882
VEL_LIMITE_AEREA_GENERAL_MS = 20.0
VEL_MAX_MS = 10.0  # límite práctico adoptado para instalación común/receptora según consulta GLP
VELOCIDAD_OBJ_MIN_MS = 8.0
VELOCIDAD_OBJ_MAX_MS = 10.0
VELOCIDAD_OBJ_CENTRO_MS = 9.5
BASE_CALCULOS = Path('.').resolve()
RAIZ_PROYECTO = BASE_CALCULOS.parent
SALIDA_ANOTACION = RAIZ_PROYECTO / 'Proyecto' / 'Anotaciones' / 'dimensionado_tuberias_glp.md'

criterio_materiales = '''La red se proyecta para GLP en fase gas y trazado exterior aereo. Se adopta tuberia metalica de cobre duro estirado sin soldadura conforme a EN 1057, con espesor minimo de 1 mm, por su compatibilidad con instalaciones receptoras de GLP y por permitir ejecucion vista protegida. El polietileno se descarta para estos tramos al tratarse de conduccion aerea expuesta; el acero al carbono se considera alternativa tecnica posible si se justifica proteccion frente a corrosion y autorizacion/criterio especifico aplicable a media presion.'''

criterio_velocidades = '''El dimensionado se realiza en media presion con presion inicial relativa de 1,70 bar y caida maxima admisible del 5%, por lo que la presion minima de comprobacion es 1,615 bar relativos. Se adopta una velocidad practica de diseno no superior a 10 m/s para limitar ruido y perdidas localizadas, manteniendo 20 m/s como limite absoluto para conducciones aereas generales. El rango operativo preferente se fija en 8-10 m/s; cuando no es alcanzable por diametro comercial minimo o por equilibrio de presion, la velocidad baja se documenta como condicion justificada, no como no conformidad.'''

material_seleccionado = 'Cobre duro estirado sin soldadura EN 1057, espesor minimo 1 mm, instalacion aerea protegida'
justificacion_material = (
    'Se selecciona cobre duro estirado sin soldadura como material metalico apto para la red exterior en fase gas '
    'para fase gas en GLP y evita la restriccion indicada para redes de acero en media presion salvo autorizacion. '
    'El PE se descarta por ser canalizacion aerea exterior. La alternativa de acero al carbono queda tecnicamente posible '
    'solo con proteccion pasiva y autorizacion o documentacion especifica. La seleccion queda supeditada a que el material '
    'mantenga presion maxima admisible suficiente frente a la presion de servicio y a las condiciones de montaje exterior.'
)

catalogo_diametros = pd.DataFrame([
    {'material': 'cobre_duro_en1057', 'designacion': '15x1', 'D_int_mm': 13.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '18x1', 'D_int_mm': 16.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '22x1', 'D_int_mm': 20.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '28x1', 'D_int_mm': 26.0, 'espesor_mm': 1.0},
    {'material': 'cobre_duro_en1057', 'designacion': '35x1.5', 'D_int_mm': 32.0, 'espesor_mm': 1.5},
    {'material': 'cobre_duro_en1057', 'designacion': '42x1.5', 'D_int_mm': 39.0, 'espesor_mm': 1.5},
    {'material': 'cobre_duro_en1057', 'designacion': '54x2', 'D_int_mm': 50.0, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '64x2', 'D_int_mm': 60.0, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '76.1x2', 'D_int_mm': 72.1, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '88.9x2', 'D_int_mm': 84.9, 'espesor_mm': 2.0},
    {'material': 'cobre_duro_en1057', 'designacion': '108x2.5', 'D_int_mm': 103.0, 'espesor_mm': 2.5},
]).sort_values('D_int_mm').reset_index(drop=True)

coef_accesorios = {'codo_90': 30, 'codo_45': 15, 'te_linea': 20, 'te_desviada': 60, 'valvula_corte': 10, 'reduccion': 10}
accesorios_base = {
    'D1-D4': {'codo_90': 1, 'valvula_corte': 1}, 'D2-D4': {'codo_90': 1, 'valvula_corte': 1},
    'D3-D4': {'codo_90': 1, 'valvula_corte': 1}, 'D4 vertical': {'codo_90': 1, 'valvula_corte': 1},
    'D4-A': {'codo_90': 1, 'valvula_corte': 1, 'te_linea': 1},
    'A-C1': {'te_desviada': 1, 'valvula_corte': 1, 'reduccion': 1}, 'A-B': {'codo_90': 1, 'te_linea': 1},
    'B-C2': {'te_desviada': 1, 'valvula_corte': 1, 'reduccion': 1}, 'B-C': {'codo_90': 1, 'te_linea': 1},
    'C-C3': {'te_desviada': 1, 'valvula_corte': 1, 'reduccion': 1}, 'C-D': {'codo_90': 1, 'te_linea': 1},
    'D-C4': {'te_desviada': 1, 'valvula_corte': 1, 'reduccion': 1}, 'D-E': {'codo_90': 1, 'te_linea': 1},
    'E-C5': {'te_desviada': 1, 'valvula_corte': 1, 'reduccion': 1}, 'E-C6': {'te_desviada': 1, 'valvula_corte': 1, 'reduccion': 1},
}

def tabla_md(df, cols=None, floatfmt='.3f'):
    data = df.copy() if cols is None else df[cols].copy()
    for col in data.select_dtypes(include=['float', 'float64']).columns:
        data[col] = data[col].map(lambda x: '' if pd.isna(x) else format(x, floatfmt))
    data = data.fillna('').astype(str)
    headers = list(data.columns)
    rows = data.values.tolist()
    def clean(value):
        return value.replace('|', '\\|').replace('\n', ' ')
    return '\n'.join([
        '| ' + ' | '.join(clean(h) for h in headers) + ' |',
        '| ' + ' | '.join('---' for _ in headers) + ' |',
        *['| ' + ' | '.join(clean(v) for v in row) + ' |' for row in rows],
    ])

def longitud_calculo(L_real_m, D_mm, accesorios):
    total = float(L_real_m)
    for nombre, cantidad in accesorios.items():
        total += cantidad * coef_accesorios[nombre] * (D_mm / 1000.0)
    return total

def renouard_mp(PA_abs_bar, Q_m3_h, D_mm, Lc_m, dc=DC_PROPANO):
    termino = 51.5 * dc * Lc_m * (Q_m3_h ** 1.82) / (D_mm ** 4.82)
    pb2 = PA_abs_bar ** 2 - termino
    if pb2 <= 0:
        return None, None, termino
    PB_abs_bar = sqrt(pb2)
    return PB_abs_bar, PA_abs_bar - PB_abs_bar, termino

def velocidad_gas(Q_m3_h, P_abs_bar, D_mm):
    return 378.04 * Q_m3_h / (P_abs_bar * (D_mm ** 2))

tramos_csv = pd.read_csv('datos/longitudes_Tramos.csv', sep=';').dropna(how='all').copy()
tramos_csv = tramos_csv.rename(columns={'Nº': 'numero', 'Tramo / zona': 'tramo_zona', 'Designación': 'designacion', 'Longitud (m)': 'longitud_m'})
tramos_csv['numero'] = tramos_csv['numero'].astype(int)
tramos_csv['longitud_m'] = pd.to_numeric(tramos_csv['longitud_m'], errors='coerce')
tramos_csv['designacion'] = tramos_csv['designacion'].astype(str).str.strip()
tramos_red = tramos_csv[tramos_csv['numero'] != 1].copy()
tramos_red[['nodo_ini', 'nodo_fin']] = tramos_red['designacion'].str.split('-', expand=True)
tramos_red['tipo_tramo'] = 'red_principal'
depositos_verticales = pd.DataFrame([
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D1', 'designacion': 'D1-D4', 'longitud_m': 1.90, 'nodo_ini': 'D1', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D2', 'designacion': 'D2-D4', 'longitud_m': 1.90, 'nodo_ini': 'D2', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D3', 'designacion': 'D3-D4', 'longitud_m': 1.90, 'nodo_ini': 'D3', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
    {'numero': 1, 'tramo_zona': 'Derivacion vertical deposito D4', 'designacion': 'D4 vertical', 'longitud_m': 1.90, 'nodo_ini': 'D4_dep', 'nodo_fin': 'D4', 'tipo_tramo': 'derivacion_deposito'},
])
df_tramos_limpio = pd.concat([depositos_verticales, tramos_red], ignore_index=True)
df_tramos_limpio = df_tramos_limpio[['numero', 'tramo_zona', 'designacion', 'longitud_m', 'nodo_ini', 'nodo_fin', 'tipo_tramo']]

df_consumidores_red = df_cons.copy().reset_index(drop=True)
df_consumidores_red['nodo'] = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']
df_consumidores_red['q_kg_h'] = df_consumidores_red['potencia_kw'] / PCS_PROPANO
df_consumidores_red['q_m3_h'] = df_consumidores_red['q_kg_h'] / DENSIDAD_PROPANO_GAS_KG_M3
q_total_kg_h = df_consumidores_red['q_kg_h'].sum()
q_total_m3_h = df_consumidores_red['q_m3_h'].sum()
children = {}
for _, row in tramos_red.iterrows():
    children.setdefault(row['nodo_ini'], []).append(row['nodo_fin'])
consumo_por_nodo = df_consumidores_red.set_index('nodo')['q_m3_h'].to_dict()
@lru_cache(None)
def caudal_descendente(nodo):
    return consumo_por_nodo.get(nodo, 0.0) + sum(caudal_descendente(hijo) for hijo in children.get(nodo, []))
q_por_designacion = {row['designacion']: caudal_descendente(row['nodo_fin']) for _, row in tramos_red.iterrows()}
for designacion in depositos_verticales['designacion']:
    q_por_designacion[designacion] = q_total_m3_h / len(depositos_verticales)
df_tramos_limpio['q_m3_h'] = df_tramos_limpio['designacion'].map(q_por_designacion)
df_tramos_limpio['q_kg_h'] = df_tramos_limpio['q_m3_h'] * DENSIDAD_PROPANO_GAS_KG_M3

filas_acc = []
for designacion, accesorios in accesorios_base.items():
    for accesorio, cantidad in accesorios.items():
        filas_acc.append({'designacion': designacion, 'accesorio': accesorio, 'cantidad': cantidad, 'coef_leq_D': coef_accesorios[accesorio], 'criterio': 'visible/hipotesis conservadora segun esquema_instalacion.pdf'})
df_accesorios = pd.DataFrame(filas_acc)

def diametro_exterior_mm(designacion):
    return float(str(designacion).split('x')[0])

catalogo_diametros['D_ext_mm'] = catalogo_diametros['designacion'].map(diametro_exterior_mm)
catalogo_diametros['area_metal_mm2'] = np.pi / 4.0 * (catalogo_diametros['D_ext_mm'] ** 2 - catalogo_diametros['D_int_mm'] ** 2)
catalogo_diametros['masa_lineal_kg_m'] = catalogo_diametros['area_metal_mm2'] * 1e-6 * 8960.0


def evaluar_candidatos_tramo(tramo, P_ini_abs_bar, catalogo):
    accesorios = accesorios_base.get(tramo['designacion'], {})
    Q = float(tramo['q_m3_h'])
    candidatos = []
    for _, tuberia in catalogo.iterrows():
        D = float(tuberia['D_int_mm'])
        Lc = longitud_calculo(tramo['longitud_m'], D, accesorios)
        q_d = Q / D if D else float('inf')
        P_fin_abs, dp_bar, _ = renouard_mp(P_ini_abs_bar, Q, D, Lc)
        masa_tramo = float(tuberia['masa_lineal_kg_m']) * float(tramo['longitud_m'])
        fila = {
            'designacion_tubo': tuberia['designacion'], 'D_int_mm': D, 'D_ext_mm': float(tuberia['D_ext_mm']),
            'masa_lineal_kg_m': float(tuberia['masa_lineal_kg_m']), 'masa_tramo_kg': masa_tramo,
            'Lc_m': Lc, 'Q_D': q_d, 'velocidad_objetivo': f'{VELOCIDAD_OBJ_MIN_MS:.0f}-{VELOCIDAD_OBJ_MAX_MS:.0f} m/s'
        }
        if P_fin_abs is None:
            fila.update({'P_fin_abs_bar': np.nan, 'P_fin_rel_bar': np.nan, 'delta_p_bar': np.nan, 'velocidad_ms': np.nan, 'desviacion_velocidad_ms': np.nan, 'cumple_Q_D': q_d < 150, 'cumple_presion': False, 'cumple_velocidad_practica': False, 'cumple_velocidad_aerea': False, 'criterio_seleccion': 'rechazado: presion final no real', 'estado': 'No conforme'})
        else:
            v = velocidad_gas(Q, P_fin_abs, D)
            cumple_qd = q_d < 150
            cumple_p = P_fin_abs >= P_MIN_ABS_BAR
            cumple_v_practica = v <= VEL_MAX_MS
            cumple_v_aerea = v <= VEL_LIMITE_AEREA_GENERAL_MS
            en_objetivo = VELOCIDAD_OBJ_MIN_MS <= v <= VELOCIDAD_OBJ_MAX_MS
            fila.update({'P_fin_abs_bar': P_fin_abs, 'P_fin_rel_bar': P_fin_abs - P_ATM_BAR, 'delta_p_bar': dp_bar, 'velocidad_ms': v, 'desviacion_velocidad_ms': abs(v - VELOCIDAD_OBJ_CENTRO_MS), 'cumple_Q_D': cumple_qd, 'cumple_presion': cumple_p, 'cumple_velocidad_practica': cumple_v_practica, 'cumple_velocidad_aerea': cumple_v_aerea, 'criterio_seleccion': 'candidato', 'estado': 'Candidato' if (cumple_qd and cumple_p and cumple_v_aerea) else 'No conforme'})
            if cumple_qd and cumple_p and cumple_v_aerea:
                if en_objetivo:
                    fila['estado'] = 'Conforme - velocidad objetivo'
                    fila['criterio_seleccion'] = 'dentro del rango practico 8-10 m/s'
                elif v < VELOCIDAD_OBJ_MIN_MS:
                    fila['estado'] = 'Conforme - velocidad baja justificada'
                    fila['criterio_seleccion'] = 'por debajo del rango; revisar si catalogo/presion permite menor diametro'
                elif cumple_v_practica:
                    fila['estado'] = 'Conforme - velocidad admisible bajo limite practico'
                    fila['criterio_seleccion'] = 'menor desviacion frente al centro objetivo sin superar 10 m/s'
                else:
                    fila['estado'] = 'Conforme - velocidad alta admisible por limite aereo general'
                    fila['criterio_seleccion'] = 'supera 10 m/s; solo admisible si se justifica como red general aerea'
        candidatos.append(fila)
    return pd.DataFrame(candidatos)


def dimensionar_tramo(tramo, P_ini_abs_bar, catalogo):
    df_candidatos = evaluar_candidatos_tramo(tramo, P_ini_abs_bar, catalogo)
    viables = df_candidatos[(df_candidatos['cumple_Q_D']) & (df_candidatos['cumple_presion']) & (df_candidatos['cumple_velocidad_aerea'])].copy()
    if viables.empty:
        fallo = df_candidatos.iloc[-1].to_dict()
        fallo['estado'] = 'No conforme'
        fallo['criterio_seleccion'] = 'ningun diametro cumple simultaneamente presion, Q/D y limite aereo'
        return fallo, df_candidatos
    objetivo = viables[(viables['velocidad_ms'] >= VELOCIDAD_OBJ_MIN_MS) & (viables['velocidad_ms'] <= VELOCIDAD_OBJ_MAX_MS)].copy()
    if not objetivo.empty:
        elegido = objetivo.sort_values(['desviacion_velocidad_ms', 'D_int_mm']).iloc[0].to_dict()
        elegido['estado'] = 'Conforme - velocidad objetivo'
        elegido['criterio_seleccion'] = 'diametro con velocidad dentro de 8-10 m/s mas cercana a 9.5 m/s'
        return elegido, df_candidatos
    practica = viables[viables['velocidad_ms'] <= VEL_MAX_MS].copy()
    if not practica.empty:
        elegido = practica.sort_values(['desviacion_velocidad_ms', 'D_int_mm']).iloc[0].to_dict()
        elegido['estado'] = 'Conforme - velocidad baja justificada' if elegido['velocidad_ms'] < VELOCIDAD_OBJ_MIN_MS else 'Conforme - velocidad admisible bajo limite practico'
        elegido['criterio_seleccion'] = 'diametro mas cercano al rango objetivo sin superar el limite practico de 10 m/s'
        return elegido, df_candidatos
    elegido = viables.sort_values(['desviacion_velocidad_ms', 'D_int_mm']).iloc[0].to_dict()
    elegido['estado'] = 'Conforme - velocidad alta admisible por limite aereo general'
    elegido['criterio_seleccion'] = 'no hay candidato <=10 m/s; se mantiene bajo limite aereo absoluto de 20 m/s'
    return elegido, df_candidatos


def optimizar_subarbol_E(P_ini_abs_bar):
    tramos_sub = {d: df_tramos_limpio[df_tramos_limpio['designacion'] == d].iloc[0].to_dict() for d in ['D-E', 'E-C5', 'E-C6']}
    combinaciones = []
    candidatos_de = evaluar_candidatos_tramo(tramos_sub['D-E'], P_ini_abs_bar, catalogo_diametros)
    for _, de in candidatos_de.iterrows():
        if not (de['cumple_Q_D'] and de['cumple_presion'] and de['cumple_velocidad_practica']):
            continue
        candidatos_c5 = evaluar_candidatos_tramo(tramos_sub['E-C5'], de['P_fin_abs_bar'], catalogo_diametros)
        candidatos_c6 = evaluar_candidatos_tramo(tramos_sub['E-C6'], de['P_fin_abs_bar'], catalogo_diametros)
        viables_c5 = candidatos_c5[(candidatos_c5['cumple_Q_D']) & (candidatos_c5['cumple_presion']) & (candidatos_c5['cumple_velocidad_practica'])].copy()
        viables_c6 = candidatos_c6[(candidatos_c6['cumple_Q_D']) & (candidatos_c6['cumple_presion']) & (candidatos_c6['cumple_velocidad_practica'])].copy()
        for _, c5 in viables_c5.iterrows():
            for _, c6 in viables_c6.iterrows():
                masa_total = de['masa_tramo_kg'] + c5['masa_tramo_kg'] + c6['masa_tramo_kg']
                desviacion_total = de['desviacion_velocidad_ms'] + c5['desviacion_velocidad_ms'] + c6['desviacion_velocidad_ms']
                combinaciones.append({
                    'D-E_tubo': de['designacion_tubo'], 'E-C5_tubo': c5['designacion_tubo'], 'E-C6_tubo': c6['designacion_tubo'],
                    'D-E_velocidad_ms': de['velocidad_ms'], 'E-C5_velocidad_ms': c5['velocidad_ms'], 'E-C6_velocidad_ms': c6['velocidad_ms'],
                    'D-E_P_fin_rel_bar': de['P_fin_rel_bar'], 'E-C5_P_fin_rel_bar': c5['P_fin_rel_bar'], 'E-C6_P_fin_rel_bar': c6['P_fin_rel_bar'],
                    'masa_total_kg': masa_total, 'desviacion_total_ms': desviacion_total,
                    'D-E_resultado': de.to_dict(), 'E-C5_resultado': c5.to_dict(), 'E-C6_resultado': c6.to_dict(),
                    'D-E_candidatos': candidatos_de, 'E-C5_candidatos': candidatos_c5, 'E-C6_candidatos': candidatos_c6,
                })
    df_combinaciones = pd.DataFrame(combinaciones)
    if df_combinaciones.empty:
        return None, df_combinaciones
    df_combinaciones = df_combinaciones.sort_values(['masa_total_kg', 'desviacion_total_ms']).reset_index(drop=True)
    return df_combinaciones.iloc[0].to_dict(), df_combinaciones

orden_depositos = ['D1-D4', 'D2-D4', 'D3-D4', 'D4 vertical']
orden_red_previa_E = ['D4-A', 'A-C1', 'A-B', 'B-C2', 'B-C', 'C-C3', 'C-D', 'D-C4']
presion_nodo = {'D4': P_INICIAL_ABS_BAR}
resultados = []
candidatos_por_tramo = {}
for designacion in orden_depositos:
    tramo = df_tramos_limpio[df_tramos_limpio['designacion'] == designacion].iloc[0].to_dict()
    elegido, candidatos = dimensionar_tramo(tramo, P_INICIAL_ABS_BAR, catalogo_diametros)
    candidatos_por_tramo[designacion] = candidatos
    resultados.append({**tramo, **elegido, 'P_ini_abs_bar': P_INICIAL_ABS_BAR, 'P_ini_rel_bar': P_INICIAL_REL_BAR})
for designacion in orden_red_previa_E:
    tramo = df_tramos_limpio[df_tramos_limpio['designacion'] == designacion].iloc[0].to_dict()
    P_ini = presion_nodo[tramo['nodo_ini']]
    elegido, candidatos = dimensionar_tramo(tramo, P_ini, catalogo_diametros)
    candidatos_por_tramo[designacion] = candidatos
    resultados.append({**tramo, **elegido, 'P_ini_abs_bar': P_ini, 'P_ini_rel_bar': P_ini - P_ATM_BAR})
    if not np.isnan(elegido.get('P_fin_abs_bar', np.nan)):
        presion_nodo[tramo['nodo_fin']] = elegido['P_fin_abs_bar']

optimizacion_subarbol_E, df_optimizacion_subarbol_E = optimizar_subarbol_E(presion_nodo['D'])
if optimizacion_subarbol_E is None:
    orden_red_final = ['D-E', 'E-C5', 'E-C6']
    for designacion in orden_red_final:
        tramo = df_tramos_limpio[df_tramos_limpio['designacion'] == designacion].iloc[0].to_dict()
        P_ini = presion_nodo[tramo['nodo_ini']]
        elegido, candidatos = dimensionar_tramo(tramo, P_ini, catalogo_diametros)
        candidatos_por_tramo[designacion] = candidatos
        resultados.append({**tramo, **elegido, 'P_ini_abs_bar': P_ini, 'P_ini_rel_bar': P_ini - P_ATM_BAR})
        if not np.isnan(elegido.get('P_fin_abs_bar', np.nan)):
            presion_nodo[tramo['nodo_fin']] = elegido['P_fin_abs_bar']
else:
    for designacion in ['D-E', 'E-C5', 'E-C6']:
        tramo = df_tramos_limpio[df_tramos_limpio['designacion'] == designacion].iloc[0].to_dict()
        resultado = optimizacion_subarbol_E[f'{designacion}_resultado'].copy()
        candidatos_por_tramo[designacion] = optimizacion_subarbol_E[f'{designacion}_candidatos']
        if designacion == 'D-E':
            P_ini = presion_nodo['D']
            presion_nodo['E'] = resultado['P_fin_abs_bar']
            resultado['criterio_seleccion'] = 'optimizacion economica del subarbol E: aumenta D-E para reducir masa total en E-C6'
            resultado['estado'] = 'Conforme - velocidad baja justificada' if resultado['velocidad_ms'] < VELOCIDAD_OBJ_MIN_MS else resultado['estado']
        else:
            P_ini = presion_nodo['E']
            resultado['criterio_seleccion'] = 'seleccionado por optimizacion economica del subarbol E con presion aguas arriba recalculada'
            if VELOCIDAD_OBJ_MIN_MS <= resultado['velocidad_ms'] <= VELOCIDAD_OBJ_MAX_MS:
                resultado['estado'] = 'Conforme - velocidad objetivo'
        resultados.append({**tramo, **resultado, 'P_ini_abs_bar': P_ini, 'P_ini_rel_bar': P_ini - P_ATM_BAR})
        if not np.isnan(resultado.get('P_fin_abs_bar', np.nan)):
            presion_nodo[tramo['nodo_fin']] = resultado['P_fin_abs_bar']

columnas_dimensionado = ['numero', 'tipo_tramo', 'tramo_zona', 'designacion', 'nodo_ini', 'nodo_fin', 'longitud_m', 'q_m3_h', 'q_kg_h', 'designacion_tubo', 'D_int_mm', 'Lc_m', 'Q_D', 'P_ini_rel_bar', 'P_fin_rel_bar', 'delta_p_bar', 'velocidad_ms', 'velocidad_objetivo', 'desviacion_velocidad_ms', 'cumple_Q_D', 'cumple_presion', 'cumple_velocidad_practica', 'cumple_velocidad_aerea', 'criterio_seleccion', 'estado']
df_dimensionado = pd.DataFrame(resultados)[columnas_dimensionado].copy()
df_no_conformidades = df_dimensionado[df_dimensionado['estado'].str.startswith('No conforme')].copy()
resumen_depositos = df_dimensionado[df_dimensionado['tipo_tramo'] == 'derivacion_deposito'].copy()
resumen_depositos_agrupado = pd.DataFrame([{'designacion': 'D1/D2/D3/D4 verticales', 'unidades': len(resumen_depositos), 'longitud_m_por_unidad': resumen_depositos['longitud_m'].iloc[0], 'q_m3_h_por_unidad': resumen_depositos['q_m3_h'].iloc[0], 'designacion_tubo': resumen_depositos['designacion_tubo'].mode().iloc[0], 'D_int_mm': resumen_depositos['D_int_mm'].max(), 'Lc_m_por_unidad': resumen_depositos['Lc_m'].max(), 'P_fin_rel_bar_min': resumen_depositos['P_fin_rel_bar'].min(), 'velocidad_ms_max': resumen_depositos['velocidad_ms'].max(), 'estado': 'Conforme' if (~resumen_depositos['estado'].str.startswith('No conforme')).all() else 'No conforme'}])
tramo_control = df_dimensionado[df_dimensionado['designacion'] == 'D4-A'].iloc[0]
control_manual = {'tramo': tramo_control['designacion'], 'Lc_recalculada_m': longitud_calculo(tramo_control['longitud_m'], tramo_control['D_int_mm'], accesorios_base[tramo_control['designacion']]), 'Q_D_recalculado': tramo_control['q_m3_h'] / tramo_control['D_int_mm'], 'velocidad_recalculada_ms': velocidad_gas(tramo_control['q_m3_h'], tramo_control['P_fin_rel_bar'] + P_ATM_BAR, tramo_control['D_int_mm'])}

def texto_dimensionado_markdown(titulo, contexto):
    cols_resultados = ['designacion', 'tipo_tramo', 'longitud_m', 'q_m3_h', 'designacion_tubo', 'D_int_mm', 'Lc_m', 'Q_D', 'P_ini_rel_bar', 'P_fin_rel_bar', 'delta_p_bar', 'velocidad_ms', 'velocidad_objetivo', 'desviacion_velocidad_ms', 'criterio_seleccion', 'estado']
    cols_consumidores = ['nodo', 'nombre', 'potencia_kw', 'q_kg_h', 'q_m3_h']
    cols_acc = ['designacion', 'accesorio', 'cantidad', 'coef_leq_D', 'criterio']
    cols_opt_e = ['D-E_tubo', 'E-C5_tubo', 'E-C6_tubo', 'D-E_velocidad_ms', 'E-C5_velocidad_ms', 'E-C6_velocidad_ms', 'D-E_P_fin_rel_bar', 'E-C5_P_fin_rel_bar', 'E-C6_P_fin_rel_bar', 'masa_total_kg', 'desviacion_total_ms']
    no_conf_text = 'No se detectan no conformidades.' if df_no_conformidades.empty else tabla_md(df_no_conformidades[cols_resultados])
    opt_e_text = 'No se genero tabla de optimizacion del subarbol E.' if df_optimizacion_subarbol_E.empty else tabla_md(df_optimizacion_subarbol_E[cols_opt_e].head(5))
    e_c6 = df_dimensionado[df_dimensionado['designacion'] == 'E-C6'].iloc[0]
    d_e = df_dimensionado[df_dimensionado['designacion'] == 'D-E'].iloc[0]
    e_c5 = df_dimensionado[df_dimensionado['designacion'] == 'E-C5'].iloc[0]
    q_c6_kg_h = df_consumidores_red[df_consumidores_red['nodo'] == 'C6']['q_kg_h'].iloc[0]
    q_c6_m3_h = df_consumidores_red[df_consumidores_red['nodo'] == 'C6']['q_m3_h'].iloc[0]
    return f'''# {titulo}

{contexto}

## Criterios tecnicos adoptados

El dimensionado de la red de distribucion de GLP se realiza para un regimen de media presion, con presion inicial relativa de `{P_INICIAL_REL_BAR:.2f} bar`. La perdida de carga admisible se limita al `{CAIDA_MAXIMA_FRACCION * 100:.1f}%` de dicha presion inicial, lo que fija una presion minima de servicio en el extremo mas desfavorable de `{P_MIN_REL_BAR:.3f} bar relativos`.

La red se calcula para propano en fase gas. Se adopta densidad corregida Renouard `dc = {DC_PROPANO}` y poder calorifico superior de `{PCS_PROPANO:.2f} kWh/kg` para transformar la potencia termica instalada en caudal masico. El paso a caudal volumetrico se realiza con densidad de propano gas `{DENSIDAD_PROPANO_GAS_KG_M3:.3f} kg/m3`.

{criterio_materiales}

{criterio_velocidades}

Desde el punto de vista de optimizacion, el diametro seleccionado no se determina unicamente por velocidad. Para cada tramo se comprueba simultaneamente la condicion de aplicacion de Renouard `Q/D < 150`, la presion minima final, la velocidad practica y el catalogo comercial disponible. En tramos terminales de pequeno caudal puede aparecer velocidad baja cuando ya se ha alcanzado el diametro comercial minimo. En ramales de mayor longitud se evalua ademas el equilibrio entre aumentar ligeramente el tramo comun aguas arriba y reducir el diametro del ramal final, usando la masa lineal del tubo como estimador tecnico del coste.

## Material y catalogo de calculo

Material adoptado: **{material_seleccionado}**.

{justificacion_material}

El catalogo utilizado incorpora diametro interior, espesor, diametro exterior y masa lineal estimada a partir de la seccion metalica y densidad del cobre `8960 kg/m3`. Esta masa lineal se emplea solo como criterio comparativo interno para seleccionar alternativas equivalentes desde el punto de vista hidraulico.

Las dimensiones comerciales y masas lineales empleadas se han contrastado con tablas de tuberia de cobre conforme a EN 1057 para aplicaciones de agua/gas y con tablas tecnicas de instalaciones de GLP. Como referencias externas de material se consideran: [EN 1057:2006+A1:2010 - seamless copper tubes for water and gas](https://standards.iteh.ai/catalog/standards/cen/472e132f-b218-4e1e-9158-a3974034b500/en-1057-2006a1-2010), [Manual de instalaciones de GLP - Cepsa](https://15f8034cdff6595cbfa1-1dd67c28d3aade9d3442ee99310d18bd.ssl.cf3.rackcdn.com/04422d24271abec52042f58a558069bf/1_09_glp_cepsa.pdf), [COPTECH - tubos de cobre duro R290 EN 1057](https://www.copper.rs/hard.html) y [ALSIMET - tabla de tubo sanitario EN 1057](https://alsimet.es/en/copper/copper-tubes/sanitary-copper). Estas referencias se usan para justificar el tipo de tubo, la nomenclatura `diametro exterior x espesor`, el uso de cobre duro y la estimacion de masa lineal; el catalogo definitivo debera validarse con el proveedor antes de presupuesto.

{tabla_md(catalogo_diametros)}

## Conexiones con otros documentos del proyecto

- [Datos de partida](../Datos.md): potencias nominales de consumidores y documentacion base del encargo.
- [Caudales de consumidores](caudales_consumidores.md): potencia maxima simultanea y base de demanda de la red.
- [Trazado de red](trazado_red.md): topologia de la red, longitudes y enlace con el plano de implantacion.
- [Valvuleria y accesorios](valvuleria_accesorios.md): criterio de valvulas de corte, tes, reducciones y elementos singulares.
- [Criterios normativos](criterios_normativos.md): marco normativo general de almacenamiento e instalacion de GLP.
- [Metodologia de longitud de calculo](../Especificaciones/metodologia-longitud-calculo.md): desarrollo metodologico de `Lc`, Renouard y comprobacion de velocidad.

## Formulacion empleada

La potencia de cada consumidor se transforma a caudal masico y volumetrico mediante:

```text
Q_kg/h = P_kW / PCS_propano
Q_m3/h = Q_kg/h / rho_propano_gas
```

Para el consumidor `C6`, de `1000 kW`, resulta:

```text
Q_kg/h = 1000 / {PCS_PROPANO:.2f} = {q_c6_kg_h:.2f} kg/h
Q_m3/h = {q_c6_kg_h:.2f} / {DENSIDAD_PROPANO_GAS_KG_M3:.3f} = {q_c6_m3_h:.2f} m3/h
```

La longitud de calculo se obtiene sumando a la longitud geometrica la longitud equivalente de accesorios:

```text
Lc(D) = Lreal + sum(n_i * K_i * D_mm / 1000)
```

Para el tramo `E-C6`, con `Lreal = {e_c6['longitud_m']:.2f} m`, diametro interior `{e_c6['D_int_mm']:.0f} mm` y accesorios equivalentes `te_desviada + valvula_corte + reduccion`, se obtiene `Lc = {e_c6['Lc_m']:.3f} m`.

La perdida de carga en media presion se calcula con Renouard:

```text
PA_abs^2 - PB_abs^2 = 51.5 * dc * Lc * Q^1.82 / D^4.82
```

Para `E-C6`, con `PA_rel = {e_c6['P_ini_rel_bar']:.3f} bar`, `Q = {e_c6['q_m3_h']:.3f} m3/h`, `D = {e_c6['D_int_mm']:.0f} mm` y `Lc = {e_c6['Lc_m']:.3f} m`, la presion final calculada es `PB_rel = {e_c6['P_fin_rel_bar']:.3f} bar`, con perdida de carga `{e_c6['delta_p_bar']:.3f} bar`.

La velocidad del gas se comprueba mediante:

```text
v = 378.04 * Q / (P_abs * D^2)
```

En el mismo tramo `E-C6`, el resultado es `v = {e_c6['velocidad_ms']:.3f} m/s`, dentro del rango operativo de `8-10 m/s`.

## Consumidores y caudales

{tabla_md(df_consumidores_red[cols_consumidores])}

## Topologia y caudales por tramo

Los caudales aguas arriba se obtienen por suma de los consumidores descendentes, sin simultaneidad adicional. La fila inicial de derivaciones verticales de deposito se trata como cuatro subtramos equivalentes de `1,90 m` y se resume agrupada en la tabla posterior.

{tabla_md(df_tramos_limpio[['designacion', 'tipo_tramo', 'nodo_ini', 'nodo_fin', 'longitud_m', 'q_m3_h', 'q_kg_h']])}

## Accesorios considerados

El inventario de accesorios se introduce como longitud equivalente proporcional al diametro. Se consideran codos por cambios de direccion, tes en derivaciones, valvulas de corte por consumidor y por salida de deposito, y reducciones cuando el ramal deriva hacia un diametro inferior.

{tabla_md(df_accesorios[cols_acc])}

## Derivaciones verticales de deposito

{tabla_md(resumen_depositos_agrupado)}

## Optimizacion hidraulica y economica del subarbol E

El tramo `E-C6` es un ramal largo con caudal relevante. El dimensionado secuencial inicial mantenia `D-E` en `35x1.5` y obligaba a seleccionar `E-C6` en `35x1.5` para cumplir presion, lo que dejaba una velocidad baja en el ramal final. Para evitar esta solucion local, se evalua de forma combinada el conjunto `D-E`, `E-C5` y `E-C6`.

La seleccion se realiza sobre combinaciones comerciales que cumplen `Q/D`, presion minima y `v <= {VEL_MAX_MS:.1f} m/s`, ordenadas por masa equivalente total de cobre. La alternativa adoptada incrementa `D-E` a `{d_e['designacion_tubo']}`, reduciendo su perdida de carga a `{d_e['delta_p_bar']:.3f} bar` y elevando la presion disponible en el nodo `E` hasta `{d_e['P_fin_rel_bar']:.3f} bar`. Con esa presion disponible, `E-C6` puede reducirse a `{e_c6['designacion_tubo']}` y queda con velocidad `{e_c6['velocidad_ms']:.3f} m/s`, dentro del rango objetivo.

{opt_e_text}

La velocidad de `D-E` queda en `{d_e['velocidad_ms']:.3f} m/s`, por debajo del rango preferente, pero se considera justificada porque el tramo comun conserva margen de presion y permite reducir material en el ramal largo `E-C6`. El tramo `E-C5` mantiene `{e_c5['designacion_tubo']}` con velocidad `{e_c5['velocidad_ms']:.3f} m/s`.

## Justificacion de velocidades bajas

Determinados tramos presentan velocidades inferiores al rango preferente `8-10 m/s`. Estos casos no se clasifican automaticamente como sobredimensionado, ya que el criterio principal es garantizar simultaneamente presion minima, condicion `Q/D < 150`, diametro comercial disponible y velocidad maxima admisible. La velocidad baja se acepta solo cuando existe una causa tecnica identificable.

- `A-C1` y `B-C2`: ambos alimentan consumidores de `60 kW`, con caudal `2.285 m3/h`. El diametro seleccionado es `15x1`, con diametro interior `13 mm`, que es el minimo del catalogo adoptado. La velocidad resultante queda en torno a `1.9 m/s`; reducir el diametro para aproximarse a `8-10 m/s` exigiria introducir un tubo no contemplado en el catalogo de calculo y con menor robustez mecanica. Por tanto, la baja velocidad se justifica por caudal reducido y diametro comercial minimo.

- `C-C3`: alimenta el consumidor de `500 kW`, con caudal `19.045 m3/h`. El tubo `22x1` proporciona `6.752 m/s`, por debajo del rango preferente pero cumpliendo presion y `Q/D`. El diametro inmediatamente inferior elevaria la velocidad, pero penalizaria la perdida de carga y reduciria el margen de presion disponible en un ramal conectado a la red principal. Se mantiene `22x1` como equilibrio entre margen hidraulico y dimension comercial.

- `C-D`: transporta el caudal remanente hacia los consumidores `C4`, `C5` y `C6`. La seleccion `42x1.5` da `7.113 m/s`, ligeramente inferior al objetivo, pero mantiene margen de presion para el subarbol final y evita trasladar una perdida excesiva a los ramales aguas abajo. La velocidad baja se considera admisible por equilibrio hidraulico de conjunto.

- `D-E`: tras la optimizacion combinada del subarbol `E`, se adopta `42x1.5`, con `6.062 m/s`. Esta reduccion de velocidad en el tramo comun permite elevar la presion en el nodo `E` y reducir `E-C6` a `28x1`, donde la velocidad pasa a `8.089 m/s`. El criterio no minimiza la velocidad de cada tramo de forma aislada, sino la solucion conjunta de presion, perdida de carga y masa equivalente de tubo.

## Resultados de dimensionado

{tabla_md(df_dimensionado[cols_resultados])}

## No conformidades

{no_conf_text}

## Control manual de trazabilidad

Como control independiente se recalcula el tramo `{control_manual['tramo']}`:

```text
Lc = {control_manual['Lc_recalculada_m']:.3f} m
Q/D = {control_manual['Q_D_recalculado']:.3f}
v = {control_manual['velocidad_recalculada_ms']:.3f} m/s
```

El resultado confirma la coherencia entre longitud de calculo, caudal, diametro y velocidad final del tramo principal de alimentacion.

## Conclusion tecnica

Con las hipotesis adoptadas, la red queda dimensionada para un caudal punta simultaneo de `{q_total_kg_h:.2f} kg/h`, equivalente a `{q_total_m3_h:.2f} m3/h`. Todos los tramos cumplen la condicion de aplicacion `Q/D < 150`, la presion minima relativa de `{P_MIN_REL_BAR:.3f} bar` y el limite practico de velocidad adoptado. Las velocidades bajas que permanecen en ramales concretos responden al diametro comercial minimo o a una decision de equilibrio hidraulico-economico, y quedan justificadas de forma expresa en la tabla de resultados.
'''

SALIDA_ANOTACION.parent.mkdir(parents=True, exist_ok=True)
SALIDA_ANOTACION.write_text(texto_dimensionado_markdown('Dimensionado de tuberias GLP', 'El presente documento desarrolla el dimensionado de la red de distribucion de GLP en fase gas, incluyendo criterios de calculo, seleccion de materiales, formulacion hidraulica, optimizacion de diametros y resultados finales por tramo.'), encoding='utf-8')
resultados_dimensionado = {'salida_anotacion': SALIDA_ANOTACION, 'q_total_kg_h': q_total_kg_h, 'q_total_m3_h': q_total_m3_h, 'material_seleccionado': material_seleccionado, 'justificacion_material': justificacion_material}
print(f'Material seleccionado: {material_seleccionado}')
print(f'Caudal total: {q_total_m3_h:.2f} m3/h ({q_total_kg_h:.2f} kg/h)')
print(f'Anotacion tecnica generada: {SALIDA_ANOTACION}')
print(f'Tramos dimensionados: {len(df_dimensionado)}')
print(f'No conformidades: {len(df_no_conformidades)}')


Material seleccionado: Cobre duro estirado sin soldadura EN 1057, espesor minimo 1 mm, instalacion aerea protegida
Caudal total: 99.79 m3/h (187.81 kg/h)
Anotacion tecnica generada: H:\Unidades compartidas\Practicas_Inst2\4_GLPs\Proyecto\Anotaciones\dimensionado_tuberias_glp.md
Tramos dimensionados: 15
No conformidades: 0


In [8]:
# 7.2. Tabla limpia de tramos y caudales
df_tramos_limpio


,numero,tramo_zona,designacion,longitud_m,nodo_ini,nodo_fin,tipo_tramo,q_m3_h,q_kg_h
0,1,Derivacion vertical deposito D1,D1-D4,1.90,D1,D4,derivacion_deposito,24.948674,46.953405
1,1,Derivacion vertical deposito D2,D2-D4,1.90,D2,D4,derivacion_deposito,24.948674,46.953405
2,1,Derivacion vertical deposito D3,D3-D4,1.90,D3,D4,derivacion_deposito,24.948674,46.953405
3,1,Derivacion vertical deposito D4,D4 vertical,1.90,D4_dep,D4,derivacion_deposito,24.948674,46.953405
4,2,Colector superior desde zona de depósitos,D4-A,15.12,D4,A,red_principal,99.794697,187.813620
5,3,Derivación hacia C1,A-C1,1.81,A,C1,red_principal,2.285375,4.301075
6,4,Bajante a C2,A-B,4.97,A,B,red_principal,97.509322,183.512545
7,5,Derivación hacia C2,B-C2,21.79,B,C2,red_principal,2.285375,4.301075
8,6,Bajante a C3,B-C,17.32,B,C,red_principal,95.223948,179.211470
9,7,Derivación hacia C3,C-C3,1.75,C,C3,red_principal,19.044790,35.842294


In [9]:
# 7.3. Consumidores asociados a C1-C6 y caudales de cálculo
df_consumidores_red


,nombre,potencia_kw,horas_dia,energia_diaria_kwh,nodo,q_kg_h,q_m3_h
0,Horno secado 1,60,12,720,C1,4.301075,2.285375
1,Horno secado 2,60,12,720,C2,4.301075,2.285375
2,Caldera vapor,500,10,5000,C3,35.842294,19.044790
3,Caldera agua caliente,300,8,2400,C4,21.505376,11.426874
4,Horno fusion,700,4,2800,C5,50.179211,26.662705
5,Horno decapado,1000,6,6000,C6,71.684588,38.089579


In [10]:
# 7.4. Inventario de accesorios por tramo
df_accesorios


,designacion,accesorio,cantidad,coef_leq_D,criterio
0,D1-D4,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
1,D1-D4,valvula_corte,1,10,visible/hipotesis conservadora segun esquema_i...
2,D2-D4,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
3,D2-D4,valvula_corte,1,10,visible/hipotesis conservadora segun esquema_i...
4,D3-D4,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
5,D3-D4,valvula_corte,1,10,visible/hipotesis conservadora segun esquema_i...
6,D4 vertical,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
7,D4 vertical,valvula_corte,1,10,visible/hipotesis conservadora segun esquema_i...
8,D4-A,codo_90,1,30,visible/hipotesis conservadora segun esquema_i...
9,D4-A,valvula_corte,1,10,visible/hipotesis conservadora segun esquema_i...


In [11]:
# 7.5. Resultados finales de dimensionado
df_dimensionado


,numero,tipo_tramo,tramo_zona,designacion,nodo_ini,nodo_fin,longitud_m,q_m3_h,q_kg_h,designacion_tubo,...,delta_p_bar,velocidad_ms,velocidad_objetivo,desviacion_velocidad_ms,cumple_Q_D,cumple_presion,cumple_velocidad_practica,cumple_velocidad_aerea,criterio_seleccion,estado
0,1,derivacion_deposito,Derivacion vertical deposito D1,D1-D4,D1,D4,1.90,24.948674,46.953405,22x1,...,0.005562,8.708164,8-10 m/s,0.791836,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
1,1,derivacion_deposito,Derivacion vertical deposito D2,D2-D4,D2,D4,1.90,24.948674,46.953405,22x1,...,0.005562,8.708164,8-10 m/s,0.791836,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
2,1,derivacion_deposito,Derivacion vertical deposito D3,D3-D4,D3,D4,1.90,24.948674,46.953405,22x1,...,0.005562,8.708164,8-10 m/s,0.791836,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
3,1,derivacion_deposito,Derivacion vertical deposito D4,D4 vertical,D4_dep,D4,1.90,24.948674,46.953405,22x1,...,0.005562,8.708164,8-10 m/s,0.791836,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
4,2,red_principal,Colector superior desde zona de depósitos,D4-A,D4,A,15.12,99.794697,187.813620,42x1.5,...,0.017975,9.202651,8-10 m/s,0.297349,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
5,3,red_principal,Derivación hacia C1,A-C1,A,C1,1.81,2.285375,4.301075,15x1,...,0.000608,1.897157,8-10 m/s,7.602843,True,True,True,True,diametro mas cercano al rango objetivo sin sup...,Conforme - velocidad baja justificada
6,4,red_principal,Bajante a C2,A-B,A,B,4.97,97.509322,183.512545,42x1.5,...,0.006862,9.014853,8-10 m/s,0.485147,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
7,5,red_principal,Derivación hacia C2,B-C2,B,C2,21.79,2.285375,4.301075,15x1,...,0.004884,1.905031,8-10 m/s,7.594969,True,True,True,True,diametro mas cercano al rango objetivo sin sup...,Conforme - velocidad baja justificada
8,6,red_principal,Bajante a C3,B-C,B,C,17.32,95.223948,179.211470,42x1.5,...,0.018386,8.864191,8-10 m/s,0.635809,True,True,True,True,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
9,7,red_principal,Derivación hacia C3,C-C3,C,C3,1.75,19.044790,35.842294,22x1,...,0.004289,6.752063,8-10 m/s,2.747937,True,True,True,True,diametro mas cercano al rango objetivo sin sup...,Conforme - velocidad baja justificada


In [12]:
# 7.6. No conformidades y control manual
print('Resumen agrupado de derivaciones de depósito:')
print(resumen_depositos_agrupado.to_string(index=False))
print('\nControl manual D4-A:')
for clave, valor in control_manual.items():
    print(f'{clave}: {valor}')

if df_no_conformidades.empty:
    print('\nNo se detectan no conformidades.')
else:
    print('\nNo conformidades:')
    print(df_no_conformidades.to_string(index=False))


Resumen agrupado de derivaciones de depósito:
           designacion  unidades  longitud_m_por_unidad  q_m3_h_por_unidad designacion_tubo  D_int_mm  Lc_m_por_unidad  P_fin_rel_bar_min  velocidad_ms_max   estado
D1/D2/D3/D4 verticales         4                    1.9          24.948674             22x1      20.0              2.7           1.694438          8.708164 Conforme

Control manual D4-A:
tramo: D4-A
Lc_recalculada_m: 17.46
Q_D_recalculado: 2.558838388943631
velocidad_recalculada_ms: 9.20265119651773

No se detectan no conformidades.


## 8. Visión final del dimensionado

Resumen final guardado en la libreta para revisión directa sin abrir los ficheros Markdown generados.


In [13]:
# 8. Visión final del dimensionado
columnas_vision = [
    'designacion', 'tipo_tramo', 'longitud_m', 'q_m3_h', 'designacion_tubo',
    'D_int_mm', 'Lc_m', 'P_ini_rel_bar', 'P_fin_rel_bar', 'velocidad_ms', 'velocidad_objetivo', 'criterio_seleccion', 'estado'
]

print('--- Visión final del dimensionado de tuberías GLP ---')
print(f"Material: {resultados_dimensionado['material_seleccionado']}")
print('Presión inicial: 1.70 bar relativos')
print('Criterio de velocidad: objetivo 8-10 m/s, límite práctico 10 m/s, límite aéreo absoluto 20 m/s')
print(f"Caudal punta total: {resultados_dimensionado['q_total_m3_h']:.2f} m3/h ({resultados_dimensionado['q_total_kg_h']:.2f} kg/h)")
print(f"Tramos dimensionados: {len(df_dimensionado)}")
print(f"No conformidades: {len(df_no_conformidades)}")
if df_no_conformidades.empty:
    print('No se detectan no conformidades.')
print(f"Anotación técnica: {resultados_dimensionado['salida_anotacion']}")

df_dimensionado[columnas_vision]


--- Visión final del dimensionado de tuberías GLP ---
Material: Cobre duro estirado sin soldadura EN 1057, espesor minimo 1 mm, instalacion aerea protegida
Presión inicial: 1.70 bar relativos
Criterio de velocidad: objetivo 8-10 m/s, límite práctico 10 m/s, límite aéreo absoluto 20 m/s
Caudal punta total: 99.79 m3/h (187.81 kg/h)
Tramos dimensionados: 15
No conformidades: 0
No se detectan no conformidades.
Anotación técnica: H:\Unidades compartidas\Practicas_Inst2\4_GLPs\Proyecto\Anotaciones\dimensionado_tuberias_glp.md


,designacion,tipo_tramo,longitud_m,q_m3_h,designacion_tubo,D_int_mm,Lc_m,P_ini_rel_bar,P_fin_rel_bar,velocidad_ms,velocidad_objetivo,criterio_seleccion,estado
0,D1-D4,derivacion_deposito,1.90,24.948674,22x1,20.0,2.70,1.700000,1.694438,8.708164,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
1,D2-D4,derivacion_deposito,1.90,24.948674,22x1,20.0,2.70,1.700000,1.694438,8.708164,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
2,D3-D4,derivacion_deposito,1.90,24.948674,22x1,20.0,2.70,1.700000,1.694438,8.708164,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
3,D4 vertical,derivacion_deposito,1.90,24.948674,22x1,20.0,2.70,1.700000,1.694438,8.708164,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
4,D4-A,red_principal,15.12,99.794697,42x1.5,39.0,17.46,1.700000,1.682025,9.202651,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
5,A-C1,red_principal,1.81,2.285375,15x1,13.0,2.85,1.682025,1.681417,1.897157,8-10 m/s,diametro mas cercano al rango objetivo sin sup...,Conforme - velocidad baja justificada
6,A-B,red_principal,4.97,97.509322,42x1.5,39.0,6.92,1.682025,1.675163,9.014853,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
7,B-C2,red_principal,21.79,2.285375,15x1,13.0,22.83,1.675163,1.670280,1.905031,8-10 m/s,diametro mas cercano al rango objetivo sin sup...,Conforme - velocidad baja justificada
8,B-C,red_principal,17.32,95.223948,42x1.5,39.0,19.27,1.675163,1.656777,8.864191,8-10 m/s,diametro con velocidad dentro de 8-10 m/s mas ...,Conforme - velocidad objetivo
9,C-C3,red_principal,1.75,19.044790,22x1,20.0,3.35,1.656777,1.652488,6.752063,8-10 m/s,diametro mas cercano al rango objetivo sin sup...,Conforme - velocidad baja justificada
